# Dataset Sampling — 30 per demographic cell

Samples 30 identities from each of the 24 demographic cells (4 ethnicities x 2 genders x 3 ages) in `final_dataset.csv`, giving 720 identities total.

Fixed random seed so re-running this notebook always picks the same 720 identities.

**Output: a CSV of the sampled rows only. No images are copied or moved** — the images stay in `generated_portraits_hashed_final/` exactly as they are. Downstream steps (FLUX generation) should read this CSV and look up each hash's image in place.

## 1. Locate and load the dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

# Search Drive for the folder containing final_dataset.csv, since the notebook
# lives in the same folder as the data.
DATASET_DIR = None
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "final_dataset.csv" in files:
        DATASET_DIR = root
        break

if DATASET_DIR is None:
    raise FileNotFoundError("Could not find final_dataset.csv anywhere under /content/drive/MyDrive")

print(f"Found dataset folder: {DATASET_DIR}")
CSV_PATH = os.path.join(DATASET_DIR, "final_dataset.csv")

FileNotFoundError: Could not find final_dataset.csv anywhere under /content/drive/MyDrive

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")
print(f"Unique hashes: {df['HASH'].nunique():,}")
df.head()

Loaded 9,000 rows
Unique hashes: 9,000


,HASH,ETHNICITY,GENDER,AGE
0,00d562183457ce02,African,Female,25
1,0138c6e4992e66bf,African,Female,25
2,0200880184f7bc70,African,Female,25
3,02ccc4dd9d55e5d5,African,Female,25
4,02f7f45434981436,African,Female,25


## 2. Confirm the 24 demographic cells

Sanity check before sampling: every (ethnicity, gender, age) combination should have enough rows to draw from.

In [ ]:
N_PER_CELL = 30

group_sizes = df.groupby(["ETHNICITY", "GENDER", "AGE"]).size()
print(f"Number of cells: {len(group_sizes)}")
print(f"Smallest cell size: {group_sizes.min()}")
print(f"Largest cell size: {group_sizes.max()}")

if group_sizes.min() < N_PER_CELL:
    raise ValueError(
        f"At least one cell has fewer than {N_PER_CELL} rows "
        f"(smallest is {group_sizes.min()}). Lower N_PER_CELL or check the data."
    )

group_sizes


Number of cells: 24
Smallest cell size: 375
Largest cell size: 375


ETHNICITY  GENDER  AGE
African    Female  25     375
                   40     375
                   55     375
           Male    25     375
                   40     375
                   55     375
Asian      Female  25     375
                   40     375
                   55     375
           Male    25     375
                   40     375
                   55     375
Caucasian  Female  25     375
                   40     375
                   55     375
           Male    25     375
                   40     375
                   55     375
Indian     Female  25     375
                   40     375
                   55     375
           Male    25     375
                   40     375
                   55     375
dtype: int64

## 3. Oversample 45 candidates per cell (fixed seed)

We pull 1.5x the target (45 instead of 30) per cell, since the earlier spot check found ~8% of images already have glasses. These 45 are CANDIDATES -- section 5 filters them down to 30 confirmed glasses-free identities per cell.

In [ ]:
RANDOM_SEED = 60  # fixed for reproducibility -- do not change once sampling is finalized
OVERSAMPLE_PER_CELL = 45  # 1.5x the target of 30, to absorb glasses-wearers we'll filter out

candidate_df = (
    df.groupby(["ETHNICITY", "GENDER", "AGE"], group_keys=False)
    .apply(lambda g: g.sample(n=OVERSAMPLE_PER_CELL, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f"Sampled {len(candidate_df):,} candidate identities total ({OVERSAMPLE_PER_CELL}/cell)")
candidate_df.head()

Sampled 1,080 candidate identities total (45/cell)


/tmp/ipykernel_743/2826221462.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=OVERSAMPLE_PER_CELL, random_state=RANDOM_SEED))


,HASH,ETHNICITY,GENDER,AGE
0,107bf9d85541cdd5,African,Female,25
1,d2192837585dbbeb,African,Female,25
2,08bf934e9e50539d,African,Female,25
3,f6fd8243135aa6a4,African,Female,25
4,3ed74443144a664f,African,Female,25


## 4. Install and run glasses-detector on all candidates

This is the same tool Popescu's own team used to filter glasses out of their DCFace-generated dataset (per their WACV 2025 paper, Appendix A): `mantasu/glasses-detector`. Purpose-built classifier, not a VLM -- fast, and already validated for exactly this problem.

In [ ]:
!pip install -q glasses-detector

In [ ]:
import glob
import time
import torch
from glasses_detector import GlassesClassifier

IMAGES_DIR = os.path.join(DATASET_DIR, "generated_portraits_hashed_final")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 'anyglasses' kind covers both transparent (prescription) and opaque (sunglasses) --
# our concern is any pre-existing eyewear, not just one type.
classifier = GlassesClassifier(kind="anyglasses", device=device)

Using device: cuda


In [ ]:
import os
import glob
import time

t0 = time.time()

# Build hash -> path lookup with ONE walk of the folder, instead of one glob() per hash.
all_image_files = glob.glob(os.path.join(IMAGES_DIR, "**", "*"), recursive=True)
all_image_files = [f for f in all_image_files if os.path.isfile(f)]
print(f"Found {len(all_image_files):,} files under {IMAGES_DIR} in {time.time() - t0:.1f}s")

hash_to_path = {}
for f in all_image_files:
    stem = os.path.splitext(os.path.basename(f))[0]
    hash_to_path[stem] = f

print(f"Built lookup for {len(hash_to_path):,} unique hashes")

candidate_df = candidate_df.copy()
candidate_df["IMAGE_PATH"] = candidate_df["HASH"].map(hash_to_path)

missing = candidate_df["IMAGE_PATH"].isna().sum()
if missing:
    print(f"WARNING: {missing} candidate hashes had no matching image file on disk.")
candidate_df = candidate_df.dropna(subset=["IMAGE_PATH"]).reset_index(drop=True)

print(f"Resolved {len(candidate_df):,} paths in {time.time() - t0:.1f}s total")

Found 12,393 files under /content/drive/MyDrive/Frugal AI Project/generated_portraits_hashed_final in 6.1s
Built lookup for 12,393 unique hashes
Resolved 1,080 paths in 6.1s total


In [ ]:
t0 = time.time()
has_glasses = classifier.predict(candidate_df["IMAGE_PATH"].tolist())
elapsed = time.time() - t0

candidate_df["HAS_GLASSES"] = pd.Series(has_glasses) == "present"
print(f"Classified {len(candidate_df):,} images in {elapsed:.1f}s ({elapsed / len(candidate_df):.3f}s/image)")
print(f"Flagged with glasses: {candidate_df['HAS_GLASSES'].sum():,} ({candidate_df['HAS_GLASSES'].mean()*100:.1f}%)")

Classified 1,080 images in 82.3s (0.076s/image)
Flagged with glasses: 185 (17.1%)


In [ ]:
print(set(has_glasses))
print(type(has_glasses[0]))

{'absent', 'present'}
<class 'str'>


## 5. Filter to glasses-free, then trim to exactly 30 per cell

If any cell has fewer than 30 glasses-free candidates after filtering, this raises an error rather than silently under-filling that cell -- rerun section 3 with a higher `OVERSAMPLE_PER_CELL` if that happens.

In [ ]:
clean_df = candidate_df[~candidate_df["HAS_GLASSES"]].reset_index(drop=True)

clean_counts = clean_df.groupby(["ETHNICITY", "GENDER", "AGE"]).size()
print("Glasses-free candidates remaining per cell:")
print(clean_counts)

short_cells = clean_counts[clean_counts < N_PER_CELL]
if len(short_cells) > 0:
    raise ValueError(
        f"{len(short_cells)} cell(s) have fewer than {N_PER_CELL} glasses-free candidates "
        f"after filtering:\n{short_cells}\n"
        f"Increase OVERSAMPLE_PER_CELL in section 3 and rerun from there."
    )

sampled_df = (
    clean_df.groupby(["ETHNICITY", "GENDER", "AGE"], group_keys=False)
    .apply(lambda g: g.sample(n=N_PER_CELL, random_state=RANDOM_SEED))
    .drop(columns=["IMAGE_PATH", "HAS_GLASSES"])
    .reset_index(drop=True)
)

print(f"\nFinal sample: {len(sampled_df):,} glasses-free identities")
sampled_df.head()

Glasses-free candidates remaining per cell:
ETHNICITY  GENDER  AGE
African    Female  25     42
                   40     39
                   55     37
           Male    25     35
                   40     37
                   55     41
Asian      Female  25     40
                   40     38
                   55     34
           Male    25     32
                   40     28
                   55     33
Caucasian  Female  25     40
                   40     40
                   55     38
           Male    25     38
                   40     40
                   55     40
Indian     Female  25     41
                   40     40
                   55     40
           Male    25     36
                   40     32
                   55     34
dtype: int64


ValueError: 1 cell(s) have fewer than 30 glasses-free candidates after filtering:
ETHNICITY  GENDER  AGE
Asian      Male    40     28
dtype: int64
Increase OVERSAMPLE_PER_CELL in section 3 and rerun from there.

In [ ]:
# Top up the short cell (Asian, Male, 40) with 2 more glasses-free candidates,
# drawn fresh from the full dataset (excluding hashes already in candidate_df).
already_seen = set(candidate_df["HASH"])
topup_pool = df[
    (df["ETHNICITY"] == "Asian") & (df["GENDER"] == "Male") & (df["AGE"] == 40)
    & (~df["HASH"].isin(already_seen))
]

topup_candidates = topup_pool.sample(n=10, random_state=RANDOM_SEED)  # pull extra, filter, keep 2 clean

topup_paths = []
for h in topup_candidates["HASH"]:
    matches = glob.glob(os.path.join(IMAGES_DIR, "**", f"{h}*"), recursive=True)
    topup_paths.append(matches[0] if matches else None)
topup_candidates = topup_candidates.copy()
topup_candidates["IMAGE_PATH"] = topup_paths
topup_candidates = topup_candidates.dropna(subset=["IMAGE_PATH"])

topup_labels = classifier.predict(topup_candidates["IMAGE_PATH"].tolist())
topup_candidates["HAS_GLASSES"] = pd.Series(topup_labels, index=topup_candidates.index) == "present"

topup_clean = topup_candidates[~topup_candidates["HAS_GLASSES"]].head(2)
print(f"Found {len(topup_clean)} glasses-free top-up candidates for Asian/Male/40")

clean_df = pd.concat([clean_df, topup_clean.drop(columns=["IMAGE_PATH", "HAS_GLASSES"])], ignore_index=True)

clean_counts = clean_df.groupby(["ETHNICITY", "GENDER", "AGE"]).size()
print(clean_counts)

Found 2 glasses-free top-up candidates for Asian/Male/40
ETHNICITY  GENDER  AGE
African    Female  25     42
                   40     39
                   55     37
           Male    25     35
                   40     37
                   55     41
Asian      Female  25     40
                   40     38
                   55     34
           Male    25     32
                   40     30
                   55     33
Caucasian  Female  25     40
                   40     40
                   55     38
           Male    25     38
                   40     40
                   55     40
Indian     Female  25     41
                   40     40
                   55     40
           Male    25     36
                   40     32
                   55     34
dtype: int64


In [ ]:
print(candidate_df.columns.tolist())

['HASH', 'ETHNICITY', 'GENDER', 'AGE', 'IMAGE_PATH', 'HAS_GLASSES']


In [ ]:
sampled_df = (
    clean_df.groupby(["ETHNICITY", "GENDER", "AGE"], group_keys=False)
    .apply(lambda g: g.sample(n=N_PER_CELL, random_state=RANDOM_SEED))
    .drop(columns=["IMAGE_PATH", "HAS_GLASSES"])
    .reset_index(drop=True)
)

print(f"\nFinal sample: {len(sampled_df):,} glasses-free identities")
sampled_df.head()


Final sample: 720 glasses-free identities


/tmp/ipykernel_743/2988528678.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=N_PER_CELL, random_state=RANDOM_SEED))


,HASH,ETHNICITY,GENDER,AGE
0,09534c3a20eaad55,African,Female,25
1,2522757e41af90a0,African,Female,25
2,be2c17990647d063,African,Female,25
3,1d24155c9370d312,African,Female,25
4,9b6ece6f3710b08b,African,Female,25


## 6. Verify the final sample is balanced

In [ ]:
check = sampled_df.groupby(["ETHNICITY", "GENDER", "AGE"]).size()
print(f"Cells in sample: {len(check)}")
print(f"Min per cell: {check.min()}, Max per cell: {check.max()}")
assert check.min() == check.max() == N_PER_CELL, "Sample is not perfectly balanced -- check for duplicate hashes or grouping bug"
print("Sample is perfectly balanced.")

assert sampled_df["HASH"].nunique() == len(sampled_df), "Duplicate hashes found in sample!"
print("No duplicate hashes.")

Cells in sample: 24
Min per cell: 30, Max per cell: 30
Sample is perfectly balanced.
No duplicate hashes.


## 7. Save the sampled list

Only the CSV of sampled rows is written. No images are touched, copied, or moved -- the full `generated_portraits_hashed_final/` folder stays exactly as-is. Downstream steps should read `sampled_dataset.csv` and look up each hash's image directly in the original folder.

In [ ]:
OUTPUT_PATH = os.path.join(DATASET_DIR, "sampled_dataset.csv")
sampled_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(sampled_df):,} rows to {OUTPUT_PATH}")

Saved 720 rows to /content/drive/MyDrive/Frugal AI Project/sampled_dataset.csv


## 8. Spot check — did filtering actually work?

The glasses-detector pass should have removed pre-existing glasses. This is a visual sanity check on that, not a substitute for it -- and it does NOT check for hats or masks, which glasses-detector doesn't cover and which we haven't confirmed are or aren't a problem in this dataset.

Displays a random grid of images from across different cells. Look for anything already present: glasses (should be gone now), hats, masks, jewelry, etc.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

IMAGES_DIR = os.path.join(DATASET_DIR, "generated_portraits_hashed_final")
N_SPOT_CHECK = 25  # how many images to eyeball

spot_check_rows = sampled_df.sample(n=N_SPOT_CHECK, random_state=RANDOM_SEED).reset_index(drop=True)

n_cols = 5
n_rows = -(-N_SPOT_CHECK // n_cols)  # ceil division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for ax, (_, row) in zip(axes, spot_check_rows.iterrows()):
    matches = glob.glob(os.path.join(IMAGES_DIR, "**", f"{row['HASH']}*"), recursive=True)
    if not matches:
        ax.set_title(f"MISSING\n{row['HASH'][:8]}", fontsize=8, color="red")
        ax.axis("off")
        continue
    img = Image.open(matches[0]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{row['ETHNICITY']}, {row['GENDER']}, {row['AGE']}", fontsize=8)
    ax.axis("off")

# Hide any unused subplot slots
for ax in axes[len(spot_check_rows):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

print(f"Displayed {len(spot_check_rows)} images. Check each for pre-existing accessories before proceeding to generation.")

Output hidden; open in https://colab.research.google.com to view.